In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 힐베르트 포락선(Envelope) 분석 — 반복 이벤트의 주기를 드러내기

빠르게 진동하는 반송파(carrier)에 **느린 변조(modulation)** 가 실린 신호는 원 파형의 FFT 만으로는 변조 주기가 잘 드러나지 않는다. 힐베르트 변환으로 얻은 **해석신호의 크기**, 즉 포락선 $|x_a(t)| = \sqrt{x(t)^2 + \hat{x}(t)^2}$ 는 고주파 반송파를 제거하고 **느린 포락(엔진 폭발 반복, 충격 반복, 베어링 결함 임펄스)** 만을 남긴다. 포락선을 다시 FFT 로 분석하면 시간 파형에서는 잘 안 보이던 **반복 이벤트 주파수** 가 명확히 드러난다.

이 노트북의 목표

1. **실습 1·2.** 엔진의 autorash 결함 시 엔진음 — 크고 작은 펄스가 번갈아 되풀이되는 주기를 포락선 스펙트럼으로 확인한다.
2. **실습 3·4.** Knocking 엔진음 — 거의 같은 간격으로 되풀이되는 충격 신호에서 포락선 스펙트럼이 충격 간격을 어떻게 드러내는지 본다.
3. **실습 5~8.** 베어링 내륜 결함(CWRU DE_IR) — 대역통과 필터 + 포락선으로 **BPFI** 피크를 검출한다 (실전 진단 맥락).

## 공통 임포트와 데이터 로드

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import utils

try:
    import librosa
except ImportError:
    %pip install librosa
    import librosa

In [ ]:
# autorash 결함 시 엔진음 / Knocking 엔진음
v_a, fs_a = librosa.load('./data/autorash.mp3', sr=None)
v_k, fs_k = librosa.load('./data/knocking.mp3',  sr=None)

# 두 신호 길이 맞춤 (autorash 를 Knocking 길이에 맞춰 잘라줌)
slice_length = len(v_k)
v_a = v_a[:slice_length]

T_a = len(v_a) / fs_a
t_a = np.arange(1/fs_a, T_a + 1/fs_a, 1/fs_a)
T_k = len(v_k) / fs_k
t_k = np.arange(1/fs_k, T_k + 1/fs_k, 1/fs_k)

print('Engine sound with autorash fault:', v_a.shape, 'fs =', fs_a)
print('Knocking                        :', v_k.shape, 'fs =', fs_k)

## 실습 1. autorash 결함 시 엔진음 — 시간 영역 포락선

이 엔진음은 수백 Hz~수 kHz 의 빠른 진동을 반송파로 하여 **수십 Hz 의 반복 주파수** 로 펄스가 되풀이된다. 먼저 시간 영역에서 원 신호 위에 힐베르트 포락선을 겹쳐 그려, 빠른 반송파의 위쪽 윤곽선이 어떻게 추적되는지 확인한다. 짧은 구간(0.60–0.72 s)을 확대해 *개별 펄스* 하나하나가 포락선 피크에 어떻게 대응되는지 본다.

In [ ]:
# Hilbert envelope of the raw waveform
v_a_env = utils.hilbert_envelope(v_a)

plt.figure(figsize=(8, 8))

plt.subplot(2, 1, 1)
plt.plot(t_a, v_a,      'C0', label='Engine sound with autorash fault')
plt.plot(t_a, v_a_env,  'C3', label='Envelope')
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Engine sound with autorash fault: time-domain waveform with envelope')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_a, v_a,      'C0')
plt.plot(t_a, v_a_env,  'C3')
plt.xlim([0.60, 0.72])
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Engine sound with autorash fault: waveform with envelope (zoom 0.60 - 0.72 s)')

plt.subplots_adjust(hspace=0.35)
plt.show()

**관찰.** 빨간 포락선 곡선이 파란 원 신호의 위쪽 윤곽선을 따라간다. 원 신호를 필터 없이 그대로 썼기 때문에 포락선도 잘게 출렁이지만, 펄스가 지나갈 때마다 포락선이 위로 솟는 것은 분명히 보인다. 전체 구간에서는 큰 펄스 12 개가 약 95 ms 간격으로 또렷하고, 그 사이 한가운데마다 작은 펄스가 포락선 띠 위로 조금 솟는다 (큰 · 작은 펄스를 모두 세면 간격은 약 47 ms). 확대 구간에는 0.62 s 와 0.71 s 의 큰 펄스와 그 사이 0.67 s 부근의 작은 펄스가 들어온다. 이 윤곽선이 다음 실습에서 주파수 영역으로 옮겨지면 이웃한 펄스 사이의 간격(약 47 ms)이 펄스 반복 주파수(약 21 Hz)의 저주파 피크로 드러난다.

---

## 실습 2. autorash 결함 시 엔진음 — 원 신호와 포락선의 FFT 비교

같은 신호의 **원 파형 스펙트럼** 과 **포락선 스펙트럼** 을 4분할로 비교한다. 위 두 패널은 원 파형을 0–5000 Hz 와 0–100 Hz 두 스케일로, 아래 두 패널은 포락선을 같은 두 스케일로 그린다. 원 스펙트럼에는 100 Hz 부근의 엔진 고조파 선과 1–5 kHz 의 넓은 성분이 있지만 펄스 반복 주파수 자리는 비어 있고, 포락선 스펙트럼은 0–100 Hz 영역에 펄스 반복 주파수와 그 배음이 뚜렷이 나타난다.

In [ ]:
# Raw waveform spectrum
F_raw_a, A_raw_a = utils.fft(v_a     - v_a.mean(),     fs_a)
# Envelope spectrum (별도 이름으로 시간영역 v_a_env 를 덮어쓰지 않도록 주의)
F_env_a, A_env_a = utils.fft(v_a_env - v_a_env.mean(), fs_a)

plt.figure(figsize=(10, 10))

plt.subplot(4, 1, 1)
plt.plot(F_raw_a, A_raw_a, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Engine sound with autorash fault: single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 2)
plt.plot(F_raw_a, A_raw_a, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Engine sound with autorash fault: single-sided FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplot(4, 1, 3)
plt.plot(F_env_a, A_env_a, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of engine sound with autorash fault: single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 4)
plt.plot(F_env_a, A_env_a, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of engine sound with autorash fault: FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplots_adjust(hspace=0.6)
plt.show()

**관찰.** 원 파형 스펙트럼 (위 두 패널) 에서 가장 높은 선은 84 · 127 · 168 Hz 부근에 서고, 1–5 kHz 에는 성분이 넓게 퍼져 있다. 0–100 Hz 확대에도 62 Hz 와 84 Hz 부근의 선이 보일 뿐 펄스 반복 주파수(약 21 Hz) 자리에는 피크가 없다. 반면 포락선 스펙트럼 (아래 두 패널) 의 0–100 Hz 확대 구간에는 약 21 Hz 에 **가장 높은 피크** 가 서고 그 정수배(약 42 · 63 Hz)에도 피크가 나타나 펄스 반복 주파수가 명확하게 드러난다. 10 Hz 와 31 Hz 부근의 피크는 큰 펄스와 작은 펄스가 번갈아 오는 약 94 ms 주기에서 나온다. 원 스펙트럼에서는 펄스 반복 주파수 자리가 비어 있지만, 포락선 스펙트럼은 **반복 이벤트 주기** 를 복조해 드러낸다.

---

## 실습 3. Knocking 엔진음 — 시간 영역 포락선

Knocking(노킹) 녹음은 짧은 충격 펄스가 거의 같은 간격으로 되풀이되는 신호다. 개별 펄스는 구조 공진을 가진해 고주파 파형으로 관찰된다. 먼저 시간 영역에서 원 신호 위에 힐베르트 포락선을 겹쳐 그려, 충격 펄스의 윤곽이 어떻게 추적되는지 본다.

In [ ]:
v_k_env = utils.hilbert_envelope(v_k)

plt.figure(figsize=(8, 8))

plt.subplot(2, 1, 1)
plt.plot(t_k, v_k,      'C0', label='Knocking')
plt.plot(t_k, v_k_env,  'C3', label='Envelope')
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Knocking time-domain waveform with envelope')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_k, v_k,      'C0')
plt.plot(t_k, v_k_env,  'C3')
plt.xlim([0.60, 0.72])
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Knocking waveform with envelope (zoom 0.60 - 0.72 s)')

plt.subplots_adjust(hspace=0.35)
plt.show()

**관찰.** 빨간 포락선이 충격 펄스의 위쪽 윤곽선을 따라간다. 전체 구간에서 큰 충격은 약 77 ms 마다 거의 정확히 되풀이되고, 큰 충격 사이에는 높이가 절반에 못 미치는 작은 충격이 대개 하나씩 끼어 있다 (확대 구간에는 0.616 s 와 0.693 s 의 큰 충격 둘과 0.66 s 부근의 작은 충격 하나가 들어온다). `autorash` 엔진음보다 충격이 짧고 간격이 고르다 — 다음 실습의 포락선 스펙트럼에서 이것이 깨끗한 피크 계열로 나타난다.

---

## 실습 4. Knocking 엔진음 — 원 신호와 포락선의 FFT 비교

같은 4분할 비교를 노킹 신호에 적용한다. 큰 충격의 간격이 약 77 ms 로 고르므로 포락선 스펙트럼의 저주파 영역에는 그 역수(약 13 Hz)와 정수배 자리에 피크가 줄지어 설 것을 예상한다. 두 신호의 차이가 같은 도구 (포락선 스펙트럼) 에서 어떻게 다르게 보이는지 확인한다.

In [ ]:
F_raw_k, A_raw_k = utils.fft(v_k     - v_k.mean(),     fs_k)
F_env_k, A_env_k = utils.fft(v_k_env - v_k_env.mean(), fs_k)

plt.figure(figsize=(10, 10))

plt.subplot(4, 1, 1)
plt.plot(F_raw_k, A_raw_k, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Knocking single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 2)
plt.plot(F_raw_k, A_raw_k, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Knocking single-sided FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplot(4, 1, 3)
plt.plot(F_env_k, A_env_k, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of Knocking single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 4)
plt.plot(F_env_k, A_env_k, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of Knocking FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplots_adjust(hspace=0.6)
plt.show()

**관찰.** 노킹 원 파형 스펙트럼 (위 두 패널) 에는 0.3–2 kHz 와 3.5–5 kHz 에 에너지가 넓게 퍼져 있고 (원 신호 에너지의 약 40 % 는 이 패널 밖 5 kHz 위에 있다), 0–100 Hz 확대에는 피크가 하나도 없어 충격의 반복 구조는 드러나지 않는다. 포락선 스펙트럼 (아래 두 패널) 의 0–100 Hz 확대 구간에는 약 13 Hz 와 그 정수배(26 · 52 · 65 · 78 Hz 등) 자리에 선명한 피크가 줄지어 선다. 가장 높은 피크가 13 Hz 가 아니라 26 Hz 인 것은 한 주기(약 77 ms) 안에 큰 충격과 작은 충격이 하나씩 있어 충격이 대략 그 절반 간격으로 오기 때문이다. 충격이 고른 간격으로 되풀이될수록 포락선 스펙트럼의 피크 계열이 깨끗해진다.

---

## 실습 5. CWRU 베어링 — 특성 주파수 (BPFI) 계산

베어링 내륜 결함은 볼이 결함부를 지날 때마다 **짧은 임펄스** 를 발생시키고, 이 임펄스는 베어링 하우징의 **고주파 공진(수 kHz)** 을 가진한다. 즉 측정 신호는 *고주파 공진 반송파 × 저주파 결함 반복(BPFI) 변조* 형태다.

CWRU 벤치마크 데이터 `data/data_fault_DE_IR.csv` — Drive End (DE) 내륜 결함, rpm ≈ 1772, $f_s = 12000$ Hz — 를 로드하고, 기대되는 결함 주파수 BPFI 를 계산한다. DE SKF6205 베어링의 BPFI 배수는 **5.4152** 이므로 **BPFI ≈ 5.4152 × (1772/60) ≈ 160 Hz** 가 기대된다. 이 값이 이후 실습 7·8 의 포락선 스펙트럼에서 검출되는지 확인하는 기준이 된다.

In [ ]:
data = np.array(pd.read_csv('./data/data_fault_DE_IR.csv'))
fs_f = 12000
v_f  = data[:, 1]
N_f  = len(v_f)
t_f  = np.arange(N_f) / fs_f

rpm       = 1772
f_shaft   = rpm / 60.0
BPFI      = 5.4152 * f_shaft   # DE SKF6205 inner-race
print(f'N = {N_f}, duration = {N_f/fs_f:.2f} s')
print(f'Shaft frequency : {f_shaft:.3f} Hz')
print(f'BPFI            : {BPFI:.3f} Hz')

**관찰.** Shaft 주파수 ≈ 29.5 Hz (1772 / 60), BPFI ≈ 160 Hz. 이 두 값이 이후 실습들의 기준선이 된다 — 원 스펙트럼 (실습 6) 에서는 두드러지지 않더라도, 공진 대역 필터링 + 포락선 (실습 7) 뒤 포락선 스펙트럼 (실습 8) 에서 BPFI 자리의 가장 높은 선으로 드러날 것이다.

---

## 실습 6. CWRU 베어링 — 원 시간파형과 원 스펙트럼

CWRU 결함 신호의 원 시간파형 (0–0.3 s 확대) 과 원 스펙트럼 (0–6000 Hz 전체 + 0–500 Hz 확대) 을 본다. 0–500 Hz 확대에 BPFI ≈ 160 Hz 위치를 점선으로 표시해, BPFI 자리의 선이 원 스펙트럼에서 두드러지는지 확인한다 — 두드러지지 않는 것이 예상한 결과이며, 그 이유가 다음 실습 (대역통과 필터 + 포락선) 의 동기가 된다.

In [ ]:
F_raw_f, A_raw_f = utils.fft(v_f - v_f.mean(), fs_f)

plt.figure(figsize=(10, 7))

plt.subplot(3, 1, 1)
plt.plot(t_f, v_f, 'C0')
plt.xlim([0.0, 0.3])
plt.xlabel('Time [sec]');  plt.ylabel('x(t)')
plt.title('Bearing inner-race fault raw waveform (zoom 0 - 0.3 s)')

plt.subplot(3, 1, 2)
plt.plot(F_raw_f, A_raw_f, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Raw spectrum (0 - 6000 Hz)')
plt.xlim([0, 6000])

plt.subplot(3, 1, 3)
plt.plot(F_raw_f, A_raw_f, 'C0')
plt.axvline(BPFI, color='C3', linestyle='--', label=f'BPFI = {BPFI:.1f} Hz')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Raw spectrum (zoom 0 - 500 Hz)')
plt.xlim([0, 500])
plt.legend()

plt.subplots_adjust(hspace=0.6)
plt.show()

**관찰.** 시간파형 확대에서는 큰 충격 펄스가 곳곳에서 튀지만 그 간격이 고르게 드러나지는 않고, 원 스펙트럼 0–6000 Hz 에서는 수 kHz 의 공진 대역이 지배적이다. 0–500 Hz 확대에서는 BPFI 점선 자리에도 작은 선이 서지만, 같은 구간에 그보다 높은 선이 여섯 개 있고 가장 높은 약 360 Hz 선은 그 약 6 배라 이 선만으로는 BPFI 를 식별할 수 없다 — 결함 에너지가 **공진 대역에 실려 있어** 저주파에서는 두드러지지 않기 때문. 다음 실습에서 이 공진 대역만 대역통과 필터로 분리해 포락선으로 복조한다.

---

## 실습 7. CWRU 베어링 — 공진 대역 대역통과 필터 + 힐베르트 포락선

CWRU DE 내륜 결함 신호의 주요 공진 대역인 **3500–5500 Hz** 를 `utils.filtering_zerophase` 로 추출한 뒤, 그 위에서 `utils.hilbert_envelope` 로 포락선을 얻는다. 시간 영역에서 대역통과 결과 (파랑) 위에 포락선 (빨강) 을 겹쳐 그리고, 포락선만 0–0.1 s 로 확대해 결함 임펄스의 반복 구조가 시각적으로 드러나는지 본다.

In [ ]:
# Resonance band bandpass + envelope
v_f_bp  = utils.filtering_zerophase(v_f, fs_f, 'band', f_low=3500, f_high=5500, order=4)
v_f_env = utils.hilbert_envelope(v_f_bp)

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(t_f, v_f_bp, 'C0', label='Bandpass 3500-5500 Hz')
plt.plot(t_f, v_f_env, 'C3', label='Envelope')
plt.xlim([0.0, 0.3])
plt.xlabel('Time [sec]');  plt.ylabel('x(t)')
plt.title('Bandpass waveform with envelope (zoom 0 - 0.3 s)')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_f, v_f_env, 'C3')
plt.xlim([0.0, 0.1])
plt.xlabel('Time [sec]');  plt.ylabel('Envelope')
plt.title('Envelope only (zoom 0 - 0.1 s)')

plt.subplots_adjust(hspace=0.5)
plt.show()

**관찰.** 대역통과 결과는 3500–5500 Hz 의 빠른 진동이고, 포락선은 그 반송파를 걷어 내고 진폭의 변화만 남긴다 — 통과 대역폭 (2 kHz) 만큼 잘게 출렁이는 성분 위로 **충격 덩어리** 가 솟는다. 포락선만 확대한 0–0.1 s 패널에서는 15 · 27 · 40 · 84 ms 부근에 큰 충격이 서고 그 사이에 작은 충격이 섞인다. 결함 주기는 약 6.25 ms (≈ 1/160 Hz) 이지만 충격의 세기가 축의 회전 각도에 따라 크게 달라지고 잡음도 섞여 있어, 덩어리 사이의 간격이 고르지 않고 이 파형에서 그 주기를 눈으로 재기는 어렵다 — 다음 실습에서 이 포락선을 주파수 분석해 BPFI 를 정량적으로 확인한다.

---

## 실습 8. CWRU 베어링 — 포락선 스펙트럼 (BPFI 배음 검출)

실습 7 의 포락선을 **평균 제거 후 FFT** 한다 (포락선은 항상 양수라 큰 DC 성분을 갖기 때문). 0–1000 Hz 와 0–500 Hz 두 스케일로 그리고, 0–500 Hz 패널에는 BPFI 의 1·2·3 배 위치를 점선 (`axvline`) 으로 표시해 결함 주파수와 배음이 일정 간격으로 줄지어 나타나는지 확인한다. 이것이 산업 현장에서 가장 널리 쓰이는 베어링 진단 파이프라인의 최종 그림이다.

In [ ]:
F_env_f, A_env_f = utils.fft(v_f_env - v_f_env.mean(), fs_f)

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(F_env_f, A_env_f, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope spectrum (0 - 1000 Hz)')
plt.xlim([0, 1000])

plt.subplot(2, 1, 2)
plt.plot(F_env_f, A_env_f, 'C3')
for k in range(1, 4):
    plt.axvline(k * BPFI, color='k', linestyle='--', alpha=0.6, zorder=0,  # 점선을 데이터 뒤에
                label=(f'k x BPFI' if k == 1 else None))
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope spectrum with BPFI harmonics (zoom 0 - 500 Hz)')
plt.xlim([0, 500])
plt.legend()

plt.subplots_adjust(hspace=0.5)
plt.show()

**관찰.** 포락선 스펙트럼에서 **BPFI ≈ 160 Hz** 의 피크가 가장 높다 (진폭 약 0.0175, 잡음 바닥의 약 43 배). 2 × BPFI (≈ 320 Hz, 약 0.003) 는 바로 옆 선들보다 조금 높을 뿐이고 3 × BPFI (≈ 480 Hz, 약 0.0015) 는 바닥의 약 3.6 배지만 주변 잡음 피크와 높이가 비슷하다 — 정수배 자리 가운데 이 신호에서 또렷한 것은 1 배뿐이다. 3 × BPFI 점선 왼쪽의 약 453 Hz 선은 BPFI 의 정수배가 아니다. 160 Hz 아래의 30 · 59 · 89 · 118 Hz 피크는 축 회전 주파수(약 29.5 Hz)의 정수배이고 42 · 71 · 101 · 130 Hz 피크는 BPFI 에서 축 회전 주파수의 1~4 배만큼 떨어진 사이드밴드다(17 Hz 도 같은 계열이다). 내륜은 축과 함께 돌아 결함이 하중을 받는 구역을 드나들기 때문에 충격 세기가 축 회전 주기로 출렁이고, 그 출렁임이 이 계열을 만든다 — 내륜 결함을 뒷받침하는 증거다. 다만 결함 주파수 자체는 가장 높은 피크가 아니라 이론이 지정한 BPFI 의 정수배 자리에서 읽는다. 실습 1–4 (autorash 결함 시 엔진음, knocking) 에서 확인한 "포락선은 반복 이벤트의 주기를 드러낸다"는 원리가 **회전기계 결함 진단의 표준 파이프라인** 으로 그대로 이어진다.

---

## 정리

- 포락선은 **고주파 반송파를 제거하고 저주파 변조 성분만 남기는 복조 연산** 이다.
- `utils.hilbert_envelope(x)` 한 줄이면 힐베르트 해석신호의 크기를 얻을 수 있다.
- 베어링 진단의 표준 절차는 **공진 대역 대역통과 필터 → 포락선 → 주파수 분석** 이며, 결함 특성 주파수(BPFI/BPFO 등)와 그 배음을 확인한다.
